In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, mean

spark = SparkSession.builder.appName('Ops').getOrCreate()


Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.UnsupportedOperationException: getSubject is not supported
	at java.base/javax.security.auth.Subject.getSubject(Subject.java:277)
	at org.apache.hadoop.security.UserGroupInformation.getCurrentUser(UserGroupInformation.java:588)
	at org.apache.spark.util.Utils$.$anonfun$getCurrentUserName$1(Utils.scala:2446)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.util.Utils$.getCurrentUserName(Utils.scala:2446)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:339)
	at org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:59)
	at java.base/jdk.internal.reflect.DirectConstructorHandleAccessor.newInstance(DirectConstructorHandleAccessor.java:62)
	at java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:499)
	at java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:483)
	at py4j.reflection.Metho

In [2]:
health_survey = spark.read.csv('/content/health_survey_v2.csv', inferSchema=True, header=True, nullValue='-')

health_survey.limit(5).toPandas()

,ID,F1,F5,F2,F1_1,F2_1,F6,F4,F3,F5_1,...,F2_9,F3_4,F4_3,F2_10,F1_7,F6_4,F4_4,F5_7,F3_5,F2_11
0,1,Somewhat Agree,Somewhat Disagree,Somewhat Agree,Somewhat Agree,Somewhat Agree,Somewhat Disagree,Somewhat Agree,Somewhat Agree,Somewhat Agree,...,Somewhat Agree,Somewhat Disagree,Neither Agree nor Disagree,Somewhat Agree,Somewhat Agree,Somewhat Agree,Somewhat Agree,Somewhat Agree,Somewhat Agree,Somewhat Agree
1,2,Somewhat Agree,Somewhat Disagree,Somewhat Agree,Somewhat Agree,Somewhat Agree,Somewhat Disagree,Somewhat Agree,Neither Agree nor Disagree,Neither Agree nor Disagree,...,Somewhat Agree,Somewhat Agree,Neither Agree nor Disagree,Somewhat Agree,Somewhat Agree,Somewhat Disagree,Neither Agree nor Disagree,Somewhat Agree,Neither Agree nor Disagree,Somewhat Agree
2,3,Strongly Agree,Neither Agree nor Disagree,Somewhat Agree,Strongly Agree,Strongly Agree,Somewhat Disagree,Somewhat Agree,Somewhat Agree,Somewhat Agree,...,Somewhat Agree,Somewhat Agree,Neither Agree nor Disagree,Somewhat Agree,Somewhat Agree,Somewhat Agree,Somewhat Agree,Strongly Agree,Strongly Disagree,Somewhat Agree
3,4,Somewhat Agree,Somewhat Agree,Strongly Agree,Somewhat Agree,Strongly Agree,Neither Agree nor Disagree,Neither Agree nor Disagree,Somewhat Disagree,Somewhat Agree,...,Somewhat Agree,Somewhat Disagree,Somewhat Agree,Somewhat Agree,Neither Agree nor Disagree,Neither Agree nor Disagree,Neither Agree nor Disagree,Somewhat Agree,Somewhat Disagree,Somewhat Agree
4,5,Strongly Agree,Strongly Disagree,Neither Agree nor Disagree,Strongly Agree,Somewhat Agree,Strongly Disagree,Strongly Agree,Somewhat Agree,Neither Agree nor Disagree,...,Somewhat Agree,Somewhat Agree,Neither Agree nor Disagree,Somewhat Agree,Somewhat Agree,Somewhat Agree,Somewhat Agree,Strongly Agree,Somewhat Disagree,Somewhat Agree


In [3]:
reversecode = spark.read.csv('/content/ReverseCodingItems_v2.csv', inferSchema=True, header=True, nullValue='-')

(reversecode := reversecode.select('Needs Reverse Coding?','Column Name')).limit(5).toPandas()


,Needs Reverse Coding?,Column Name
0,No,F1
1,Yes,F5
2,No,F2
3,No,F1_1
4,No,F2_1


In [4]:
(reg_map := {
    "Strongly Disagree": 1,
    "Somewhat Disagree": 2,
    "Neither Agree nor Disagree": 3,
    "Somewhat Agree": 4,
    "Strongly Agree": 5
})

{'Strongly Disagree': 1,
 'Somewhat Disagree': 2,
 'Neither Agree nor Disagree': 3,
 'Somewhat Agree': 4,
 'Strongly Agree': 5}

In [5]:
(rev_map := {'Somewhat Agree' :2,
 'Somewhat Disagree':4,
 'Strongly Agree':1,
 'Neither Agree nor Disagree':3,
 'Strongly Disagree':5}
)

{'Somewhat Agree': 2,
 'Somewhat Disagree': 4,
 'Strongly Agree': 1,
 'Neither Agree nor Disagree': 3,
 'Strongly Disagree': 5}

In [6]:
(q_cols:=
 health_survey.drop('ID').columns
)

['F1',
 'F5',
 'F2',
 'F1_1',
 'F2_1',
 'F6',
 'F4',
 'F3',
 'F5_1',
 'F1_2',
 'F2_2',
 'F6_1',
 'F2_3',
 'F4_1',
 'F2_4',
 'F5_2',
 'F2_5',
 'F6_2',
 'F1_3',
 'F2_6',
 'F5_3',
 'F4_2',
 'F2_7',
 'F3_1',
 'F2_8',
 'F5_4',
 'F3_2',
 'F1_4',
 'F3_3',
 'F1_5',
 'F5_5',
 'F6_3',
 'F1_6',
 'F5_6',
 'F2_9',
 'F3_4',
 'F4_3',
 'F2_10',
 'F1_7',
 'F6_4',
 'F4_4',
 'F5_7',
 'F3_5',
 'F2_11']

In [21]:
from pyspark.sql import functions as F

(health_survey
    .unpivot(values=q_cols,
             ids= 'ID',
             variableColumnName="Question",
             valueColumnName="Response"
    )
   .withColumn("reg_coding",
            F.map_from_arrays(
                F.array([F.lit(k) for k in reg_map.keys()]),
                F.array([F.lit(v) for v in reg_map.values()])
            )[F.col("Response")]
        )
   .withColumn("rev_coding",
            F.map_from_arrays(
                F.array([F.lit(k) for k in rev_map.keys()]),
                F.array([F.lit(v) for v in rev_map.values()])
            )[F.col("Response")]
        )
   .join(
         reversecode,
         on=F.col("Question") == F.col("Column Name"),
         how="left")
   .withColumn("coding", F.when(F.col('Needs Reverse Coding?') == 'Yes', F.col('rev_coding'))
                                 .otherwise(F.col('reg_coding')))
   .withColumn("q_type", F.regexp_replace("Question", r"_.*", ""))

   .groupBy('ID')
        .pivot('q_type')
        .sum('coding')

 ).limit(5).toPandas()

,ID,F1,F2,F3,F4,F5,F6
0,148,30,52,15,16,30,16
1,243,25,42,21,15,25,17
2,31,36,57,22,20,35,22
3,137,34,44,15,10,30,17
4,251,37,50,20,21,28,19
